# October 2019 clickstream

Charts over the gold star schema. The queries live in `sql/analysis.sql` and this notebook runs them through DuckDB.

Revenue is purchase price where `is_non_positive_price` is false. Category comes from `dim_product`. Brand is `fact_events.brand`. Hours are UTC. A cart pair is one `user_session` and `product_id` with at least one cart; it is abandoned when that session has no purchase of the same product.

The sentences under each chart were written from the October 2019 run. The load cell checks that revenue is still 229,933,212.63 before those sentences are read.


In [1]:
from pathlib import Path
import importlib.util

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "sql" / "analysis.sql").exists():
    ROOT = ROOT.parent

spec = importlib.util.spec_from_file_location("run_analysis", ROOT / "src" / "05_run_analysis.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)

sql_text = (ROOT / "sql" / "analysis.sql").read_text()
prelude, queries = analysis.parse_script(sql_text)
con = analysis.connect()
for statement in analysis.statements(prelude):
    analysis.execute(con, statement)

frames = {}
for name, _title, body in queries:
    frame = None
    for statement in analysis.statements(body):
        frame = analysis.execute(con, statement)
    if frame is not None:
        frames[name] = frame

kpis = frames["q01_kpis"].iloc[0]
revenue = float(kpis["revenue"])
if abs(revenue - 229_933_212.63) > 0.01:
    raise RuntimeError(f"Revenue {revenue} no longer matches the written insights")

BLUE = "#4C78A8"
ORANGE = "#F58518"
GREEN = "#54A24B"
RED = "#E45756"
PURPLE = "#B279A2"
TEAL = "#72B7B2"


def style(fig, title, height=480):
    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, x=0),
        font=dict(family="Arial, sans-serif", size=13),
        margin=dict(l=20, r=20, t=60, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        height=height,
    )
    return fig

print(
    f"events={int(kpis['events']):,}  purchases={int(kpis['purchases']):,}  "
    f"revenue={revenue:,.2f}  buyers={int(kpis['buyers']):,}"
)


events=42,418,542  purchases=742,773  revenue=229,933,212.63  buyers=347,118


The month has 42,418,542 events, 742,773 purchases, and revenue of 229,933,212.63. Average order value is 309.56. 347,118 of 3,022,290 users buy, a buyer rate of 0.114853.


In [2]:
funnel = frames["q02_session_funnel"]
nested = funnel[funnel["stage_group"] == "nested"]
fig = go.Figure(
    go.Funnel(
        y=nested["stage"],
        x=nested["sessions"],
        textinfo="value+percent initial",
        marker={"color": [BLUE, ORANGE, GREEN]},
    )
)
style(fig, "Sessions on the view, cart, and purchase path")
fig.update_layout(showlegend=False)
display(fig)


9,242,653 sessions view a product. 571,907 of them also cart (6.1877% of view sessions), and 291,500 view, cart, and purchase (3.1539%).

337,699 purchase sessions have no cart event. That is 53.6405% of the 629,560 purchase sessions. Apparel shows the same split in miniature: 26 cart events and 8,002 purchases. Lucente has 0 carts and 11,576 purchases.


In [3]:
category = frames["q03_category_l1"].sort_values("revenue")
colors = [ORANGE if name == "unknown" else BLUE for name in category["category_l1"]]
fig = go.Figure(
    go.Bar(
        y=category["category_l1"],
        x=category["revenue"],
        orientation="h",
        marker_color=colors,
        customdata=category[["revenue_share", "aov", "purchase_per_view"]],
        hovertemplate=(
            "%{y}<br>revenue %{x:,.2f}"
            "<br>share %{customdata[0]:.2%}"
            "<br>AOV %{customdata[1]:,.2f}"
            "<br>purchases per view %{customdata[2]:.4f}<extra></extra>"
        ),
    )
)
style(fig, "Revenue by category level 1", height=520)
fig.update_layout(showlegend=False, xaxis_title="Revenue", yaxis_title="")
display(fig)


Electronics is 176,445,577.86, 76.7378% of revenue, at an average order value of 417.15. The unknown category is second: 22,922,165.08, 9.9691% of revenue, from 173,411 purchases. Its average order value is 132.18, so the missing codes are a real sales bucket and a cheaper one.


In [4]:
level2 = frames["q04_category_l2"].sort_values("revenue")
colors = [ORANGE if name == "unknown" else TEAL for name in level2["category_l2"]]
fig = go.Figure(
    go.Bar(
        y=level2["category"],
        x=level2["revenue"],
        orientation="h",
        marker_color=colors,
        customdata=level2[["revenue_share", "aov"]],
        hovertemplate="%{y}<br>revenue %{x:,.2f}<br>share %{customdata[0]:.2%}<br>AOV %{customdata[1]:,.2f}<extra></extra>",
    )
)
style(fig, "Top category pairs by revenue, with the unknown row", height=560)
fig.update_layout(showlegend=False, xaxis_title="Revenue", yaxis_title="")
display(fig)


`electronics / smartphone` is 157,033,913.97, 68.2954% of the whole month, at an average order value of 464.63. The next known pair is `appliances / kitchen` at 10,533,020.00. The level-1 electronics total is mostly this one pair.


In [5]:
brands = frames["q05_brands"].sort_values("revenue")
colors = [ORANGE if name == "unknown" else BLUE for name in brands["brand"]]
fig = go.Figure(
    go.Bar(
        y=brands["brand"],
        x=brands["revenue"],
        orientation="h",
        marker_color=colors,
        customdata=brands[["revenue_share", "aov", "purchase_per_view"]],
        hovertemplate=(
            "%{y}<br>revenue %{x:,.2f}"
            "<br>share %{customdata[0]:.2%}"
            "<br>AOV %{customdata[1]:,.2f}"
            "<br>purchases per view %{customdata[2]:.4f}<extra></extra>"
        ),
    )
)
style(fig, "Top brands by event-brand revenue", height=560)
fig.update_layout(showlegend=False, xaxis_title="Revenue", yaxis_title="")
display(fig)


Apple is 111,199,495.70, 48.3617% of revenue, average order value 778.39, and 0.037890 purchases per view. Samsung is 46,402,037.35, 20.1807% of revenue, average order value 268.41, and 0.035969 purchases per view. View conversion is close, and Apple's average order value is much higher. Unknown brand ranks fourth at 8,539,747.08.


In [6]:
hourly = frames["q06_hourly"]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=hourly["hour_utc"], y=hourly["events"], name="Events", line={"color": BLUE, "width": 2}),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=hourly["hour_utc"], y=hourly["purchases"], name="Purchases", line={"color": GREEN, "width": 2}),
    secondary_y=True,
)
style(fig, "UTC hour of events and purchases")
fig.update_xaxes(title_text="Hour (UTC)", dtick=1)
fig.update_yaxes(title_text="Events", secondary_y=False)
fig.update_yaxes(title_text="Purchases", secondary_y=True)
display(fig)


Events peak at 16:00 UTC: 3,053,226 events, 7.1979% of traffic, and 35,772 purchases. Purchases peak at 08:00 UTC: 55,195, 7.4309% of purchases. Revenue peaks one hour later, at 09:00 UTC, at 17,551,669.73 (7.6334% of revenue). The store's local offset is not in the file, so these hours stay in UTC.


In [7]:
days = frames["q07_dow"]
days = days[days["day_of_week"] <= 7]
colors = [ORANGE if flag else BLUE for flag in days["is_weekend"]]
fig = go.Figure(
    go.Bar(
        x=days["day_name"],
        y=days["revenue"],
        marker_color=colors,
        customdata=days[["purchases", "revenue_share"]],
        hovertemplate="%{x}<br>revenue %{y:,.2f}<br>purchases %{customdata[0]:,}<br>share %{customdata[1]:.2%}<extra></extra>",
    )
)
style(fig, "Revenue by UTC weekday")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Revenue")
display(fig)


Wednesday is the strongest day, 37,064,086.24 and 16.1195% of revenue. Saturday and Sunday together are the weekend rollup: 59,036,422.18, 25.6755% of revenue. Weekdays are 170,896,790.45, 74.3245%. Tuesday carries the most events, 6,797,348.


In [8]:
daily = frames["q13_daily_revenue"]
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=daily["event_date"],
        y=daily["revenue"],
        name="Daily revenue",
        line={"color": GREEN, "width": 2},
    )
)
fig.add_trace(
    go.Scatter(
        x=daily["event_date"],
        y=daily["revenue_ma7_full"],
        name="7-day average",
        line={"color": PURPLE, "width": 3},
    )
)
style(fig, "Daily revenue and the 7-day average")
fig.update_layout(xaxis_title="UTC date", yaxis_title="Revenue")
display(fig)


The highest day is 2019-10-16, with 31,393 purchases and revenue of 9,747,164.72. The lowest is 2019-10-02, at 6,213,628.53. The 7-day average is blank until the window holds seven dates. On a full window it peaks on 2019-10-19 at 8,708,230.73 and ends on 2019-10-31 at 6,780,464.76.


In [9]:
bands = frames["q12_price_band_revenue"]
bands = bands[bands["band_key"] != "00_non_positive"]
fig = go.Figure()
fig.add_trace(go.Bar(name="Purchase share", x=bands["band_label"], y=bands["purchase_share"], marker_color=BLUE))
fig.add_trace(go.Bar(name="Revenue share", x=bands["band_label"], y=bands["revenue_share"], marker_color=GREEN))
style(fig, "Purchase mix and revenue mix by price band")
fig.update_layout(barmode="group", yaxis_tickformat=".0%", yaxis_title="Share of the month", xaxis_title="Purchase price")
display(fig)


Purchases at 1,000 and above are 49,403 events, 6.6512% of purchases, and 64,264,012.20 of revenue, 27.9490%. Purchases under 50 are 117,065 events, 15.7605% of purchases, and 3,447,165.29, 1.4992% of revenue. The non-positive band has 0 eligible purchases. The cuts sit across a distribution whose median purchase price is 179.84 and whose 99th percentile is 1,639.82.


In [10]:
abandon = frames["q11_abandon_price_band"]
fig = go.Figure(
    go.Bar(
        x=abandon["band_label"],
        y=abandon["abandon_rate"],
        marker_color=RED,
        customdata=abandon[["cart_pairs", "abandoned_cart_value"]],
        hovertemplate=(
            "%{x}<br>abandon rate %{y:.2%}"
            "<br>cart pairs %{customdata[0]:,}"
            "<br>abandoned value %{customdata[1]:,.2f}<extra></extra>"
        ),
    )
)
style(fig, "Cart abandonment rate by cart price band")
fig.update_layout(showlegend=False, yaxis_tickformat=".0%", yaxis_title="Abandoned pairs / cart pairs", xaxis_title="Cart price")
display(fig)


323,267 of 628,207 carted session-products are abandoned, a rate of 0.514587. Abandoned cart value is 107,866,053.25. The rate is 0.622010 under 50, where the abandoned value is 1,213,538.33. From 150 upward the rate stays between 0.493258 and 0.507742. Carts at 1,000 and above abandon at 0.503844 and hold 31,852,532.99 of abandoned value. The median abandoned cart price is 195.62, and the median purchased cart price is 223.65.


In [11]:
by_category = frames["q10_abandon_category"].sort_values("abandoned_cart_value")
fig = go.Figure(
    go.Bar(
        y=by_category["category_l1"],
        x=by_category["abandoned_cart_value"],
        orientation="h",
        marker_color=[ORANGE if name == "unknown" else RED for name in by_category["category_l1"]],
        customdata=by_category["abandon_rate"],
        hovertemplate="%{y}<br>abandoned value %{x:,.2f}<br>rate %{customdata:.2%}<extra></extra>",
    )
)
style(fig, "Abandoned cart value by category", height=520)
fig.update_layout(showlegend=False, xaxis_title="Abandoned cart value", yaxis_title="")
display(fig)


Electronics holds 89,266,881.40 of abandoned cart value, 82.7572% of the total, at a rate of 0.482476. The rate on electronics is lower than the overall 0.514587. The value is large because the carts are large. At session grain, 281,236 of 573,097 cart sessions contain no purchase at all, a rate of 0.490730.


In [12]:
freq = frames["q16_frequency_hist"]
fig = go.Figure()
fig.add_trace(go.Bar(name="Buyer share", x=freq["purchase_sessions"], y=freq["buyer_share"], marker_color=BLUE))
fig.add_trace(go.Bar(name="Revenue share", x=freq["purchase_sessions"], y=freq["revenue_share"], marker_color=GREEN))
style(fig, "Buyers and revenue by number of purchase sessions")
fig.update_layout(barmode="group", yaxis_tickformat=".0%", xaxis_title="Purchase sessions in October", yaxis_title="Share")
display(fig)


235,461 buyers, 67.8331%, have one purchase session and produce 70,139,813.69, 30.5044% of revenue. The 111,657 buyers with two or more sessions produce 159,793,398.94, 69.4956%. Inside that, the 18,449 buyers with five or more sessions are 5.3149% of buyers and 34.9755% of revenue (80,420,282.78).


In [13]:
segments = frames["q14_rfm_segments"].sort_values("revenue", ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(name="Buyer share", x=segments["segment"], y=segments["buyer_share"], marker_color=BLUE))
fig.add_trace(go.Bar(name="Revenue share", x=segments["segment"], y=segments["revenue_share"], marker_color=GREEN))
style(fig, "RFM segment size and revenue share", height=520)
fig.update_layout(barmode="group", yaxis_tickformat=".0%", xaxis_title="", yaxis_title="Share of buyers or revenue")
display(fig)


Champions are 27,453 buyers, 7.9088% of buyers, and 85,321,183.75 of revenue, 37.1069%. They average 6.38 purchase sessions and their recency runs from 0 to 10 days before 31 October.

Frequency NTILE scores 1, 2, and 3 are entirely one-session buyers, so the labels use the raw session count. Big spenders are the 21,830 one-session buyers in monetary score 5: 27,709,328.89 of revenue, average 1,269.32. Need-attention buyers last purchased 16 to 30 days before 31 October. Recency in this extract is a position inside the month.
